# 2. Data Understanding

## 2.1 Objective & Strategy
The goal is to understand the CICIDS2017 dataset (approx. 6 million network flows). 
Given the size of the dataset (803 MB in Parquet), loading the entire dataset into memory for EDA is inefficient. We will:
1. Read a representative sample to inspect data types and distributions.
2. Check for data quality issues (`NaN`, `inf`).
3. Analyze the target variable (`Label`) to quantify the class imbalance.

In [4]:
import pandas as pd
import numpy as np
import glob
import os

# On utilise un motif récursif pour chercher dans TOUS les sous-dossiers
parquet_pattern = "../data/processed/parquet/**/*.parquet"
parquet_files = glob.glob(parquet_pattern, recursive=True)

print(f"Nombre de fichiers Parquet trouvés : {len(parquet_files)}")

if len(parquet_files) == 0:
    print("⚠️ ERREUR : Aucun fichier trouvé.")
else:
    print(f"Chargement de l'échantillon (2 premiers fichiers)...")
    
    # 1. LECTURE AVEC FASTPARQUET (contourne l'erreur PyArrow sur les doublons)
    dfs = []
    for f in parquet_files[:2]:
        df_temp = pd.read_parquet(f, engine='fastparquet')
        
        # 2. SUPPRESSION DES COLONNES DUPLIQUÉES (garde la première occurrence)
        df_temp = df_temp.loc[:, ~df_temp.columns.duplicated()]
        
        dfs.append(df_temp)
    
    # 3. CONCATÉNATION
    df_sample = pd.concat(dfs, ignore_index=True)

    # 4. Nettoyage final (au cas où)
    df_sample.columns = df_sample.columns.str.strip()

    print(f"✅ Taille de l'échantillon : {df_sample.shape[0]} lignes, {df_sample.shape[1]} colonnes")
    print("Vérification des doublons de colonnes :", df_sample.columns.duplicated().sum())

Nombre de fichiers Parquet trouvés : 16
Chargement de l'échantillon (2 premiers fichiers)...


/tmp/ipykernel_14359/3092227723.py:20: Pandas4Warning: The 'fastparquet' engine is deprecated and will be removed in a future version. Use 'pyarrow' instead.
  df_temp = pd.read_parquet(f, engine='fastparquet')


ImportError: `Import fastparquet` failed. fastparquet is required for parquet support. Use pip, conda, or your preferred package management tool to install the fastparquet package.